# Downside Risk and VaR Methods

Module: Market Risk

## Lesson summary

This lab builds a reproducible workflow for downside risk, Value at Risk, and Expected Shortfall. Students compare empirical, Gaussian, Cornish-Fisher, and volatility-weighted estimates on the same portfolio return series.

## Learning objectives

By the end of this lab, students should be able to:

- distinguish volatility from downside-only risk;
- compute target semideviation and the Sortino ratio;
- estimate historical, Gaussian, and Cornish-Fisher VaR;
- explain why Expected Shortfall is more tail-sensitive than VaR;
- use EWMA volatility to make historical simulation more responsive to recent market regimes.

## Risk metric definitions

For a target return $\tau$, target semideviation measures only downside observations:

$$
\sigma_-(\tau)=\sqrt{\frac{1}{T}\sum_{t=1}^{T}\min(r_t-\tau,0)^2}.
$$

Value at Risk is a loss quantile. If $L=-r$ is the portfolio loss, then:

$$
\operatorname{VaR}_{\alpha}=Q_{1-\alpha}(L).
$$

Expected Shortfall averages losses beyond the VaR threshold:

$$
\operatorname{ES}_{\alpha}=\mathbb{E}\left[L\mid L\geq \operatorname{VaR}_{\alpha}\right].
$$

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew

from src.market_risk import (
    cornish_fisher_var,
    ewma_volatility,
    expected_shortfall,
    gaussian_var,
    historical_var,
    sortino_ratio,
    target_semideviation,
    volatility_weighted_historical_var,
)

## Simulated non-normal portfolio returns

The portfolio below includes common empirical features of financial returns: correlation, occasional negative jumps, negative skewness, and excess kurtosis. The data are synthetic so the notebook remains reproducible without network access.

In [ ]:
rng = np.random.default_rng(33)
dates = pd.bdate_range("2023-01-02", periods=750)

daily_mean = np.array([0.00035, 0.00025, 0.00012])
daily_covariance = np.array(
    [
        [0.00018, 0.00009, 0.00003],
        [0.00009, 0.00014, 0.00004],
        [0.00003, 0.00004, 0.00005],
    ]
)
base_returns = rng.multivariate_normal(daily_mean, daily_covariance, size=len(dates))
jumps = rng.binomial(1, 0.025, size=(len(dates), 1)) * rng.normal(
    -0.035,
    0.012,
    size=(len(dates), 1),
)
asset_returns = pd.DataFrame(
    base_returns + np.hstack([jumps, 0.55 * jumps, 0.20 * jumps]),
    index=dates,
    columns=["mexican_equity", "global_equity", "short_bond"],
)

weights = pd.Series(
    {"mexican_equity": 0.45, "global_equity": 0.35, "short_bond": 0.20},
    name="weight",
)
portfolio_returns = asset_returns.dot(weights).rename("portfolio_return")

asset_returns.head()

## Distribution diagnostics

In [ ]:
pd.DataFrame(
    {
        "mean": asset_returns.mean(),
        "volatility": asset_returns.std(),
        "skewness": asset_returns.apply(lambda series: skew(series, bias=False)),
        "excess_kurtosis": asset_returns.apply(
            lambda series: kurtosis(series, fisher=True, bias=False)
        ),
    }
)

In [ ]:
pd.Series(
    {
        "portfolio_mean": portfolio_returns.mean(),
        "portfolio_volatility": portfolio_returns.std(),
        "portfolio_skewness": skew(portfolio_returns, bias=False),
        "portfolio_excess_kurtosis": kurtosis(portfolio_returns, fisher=True, bias=False),
    }
)

## Downside risk

Volatility penalizes positive and negative surprises symmetrically. Semideviation focuses only on returns below the selected target.

In [ ]:
downside_report = pd.Series(
    {
        "daily_target_semideviation": target_semideviation(portfolio_returns, target=0.0),
        "annualized_target_semideviation": target_semideviation(
            portfolio_returns,
            target=0.0,
            periods_per_year=252,
        ),
        "sortino_ratio": sortino_ratio(portfolio_returns, target=0.0),
    }
)

downside_report

## VaR and Expected Shortfall

All VaR and Expected Shortfall estimates are reported as positive loss numbers. For example, a daily VaR of `0.025` means a 2.5% one-day portfolio loss threshold.

In [ ]:
alpha = 0.01

risk_table = pd.Series(
    {
        "historical_var": historical_var(portfolio_returns, alpha=alpha),
        "gaussian_var": gaussian_var(portfolio_returns, alpha=alpha),
        "cornish_fisher_var": cornish_fisher_var(
            portfolio_returns,
            alpha=alpha,
            validate_moments=False,
        ),
        "volatility_weighted_historical_var": volatility_weighted_historical_var(
            portfolio_returns,
            alpha=alpha,
            lambda_=0.94,
        ),
        "expected_shortfall": expected_shortfall(portfolio_returns, alpha=alpha),
    },
    name="daily_loss",
).to_frame()

risk_table

## EWMA volatility state

EWMA volatility is a simple way to make risk estimates respond faster after volatility shocks. The RiskMetrics convention often uses `lambda_=0.94` for daily returns.

In [ ]:
ewma_state = ewma_volatility(portfolio_returns, lambda_=0.94)

pd.DataFrame(
    {
        "portfolio_return": portfolio_returns,
        "ewma_volatility": ewma_state,
    }
).tail()

## Interpretation checklist

| Question | What to inspect |
| --- | --- |
| Is the distribution symmetric? | Skewness and large negative jumps |
| Is Gaussian VaR plausible? | Difference between Gaussian and historical VaR |
| Is tail loss material? | Gap between VaR and Expected Shortfall |
| Is the latest volatility regime unusual? | EWMA volatility relative to unconditional volatility |
| Is Cornish-Fisher stable? | Whether skewness and kurtosis are within a defensible range |

## Model limitations

- VaR depends strongly on the selected horizon, confidence level, sign convention, and return distribution.
- Historical methods reuse past losses and can miss new risks when market structure changes.
- Expected Shortfall is more tail-sensitive than VaR, but it still inherits the sample and model assumptions used to estimate the tail.